## Run evaluations

This notebook shows examples on how to run parsing and chunking evaluations

### Parsing evaluation (speed and energy consumption)

In [ ]:
from src.utils import get_pdf_filepaths
from src.evaluation.parsing_evaluator import ParsingEvaluator
from src.pipelines.chunknorris_pipeline import ChunkNorrisPipeline # or any other pipeline

In [ ]:
filepaths = get_pdf_filepaths("path/to/folder")
pipeline = ChunkNorrisPipeline()
parsing_eval = ParsingEvaluator(pipeline) # results will be saved in "./results" folder by default
parsing_eval.evaluate_parsing(filepaths)

### Chunking evaluation (recall and NDCG)
The ``ChunkingEvaluator`` allows to easily evaluate the best chunking strategy for a given pipeline.

Given a pipeline, we can evaluate recall on a given dataset:
- provided a list of chunkers
- provided a list of retrievers

Thus, it enables choosing the **best combination** of **parser - chunker - retriever** for maximum result.

The following examples shows how to proceed.

In [ ]:
import datasets
from src.evaluation.chunking_evaluator import ChunkingEvaluator
from src.pipelines.chunknorris_pipeline import ChunkNorrisPipeline # or any available pipeline used for parsing
from src.chunkers.page_chunker import PageChunker # or any available chunker
# Import retrievers you want to use
from src.retrievers.dense_retriever import DenseRetriever 
from src.retrievers.bm25_retriever import BM25Retriever
from src.retrievers.hybrid_retriever import HybridRetriever
# Import rerankers you want to use
from src.rerankers.crossencoder_reranker import CrossEncoderReranker
from src.rerankers.mxbai_reranker import MixBreadAIReranker

from src.utils import get_pdf_filepaths

from sentence_transformers import SentenceTransformer

#### Get the data ready:

To evaluate the pdf parsing / chunking strategy that works best for information retrieval, we need a dataset of labeled queries and their relevant passages in the documents.

The [PIRE](https://huggingface.co/datasets/Wikit/PIRE) dataset provides such data. Before going further, it is advise to have a look at the dataset card to understand it's structure.

Additionally, YOU NEED TO download the [raw pdf files](https://huggingface.co/datasets/Wikit/PIRE/blob/main/pdf_files.zip) of the dataset. They will be fed to the parser-chunkers.

You may want to use any custom dataset formatted as PIRE.

**WARNING** : DO NOT MODIFY the file names as they are used as identifiers the check the source of the chunks ! 

In [ ]:
# Get the dataset ready for evaluation !
queries_dataset = datasets.load_dataset("Wikit/PIRE")["chunk.multi"] # or chunk.single
pdf_filepaths = get_pdf_filepaths("path/to/the/folder/with_pdfs")

In [ ]:
# Load the model we will use for the DenseRetriever
model = SentenceTransformer("Snowflake/snowflake-arctic-embed-m-v2.0", trust_remote_code=True)

In [ ]:
# Prepare the evaluator
# Choose a pipeline, a list of chunkers, a list of retrievers, and a list of rerankers.
chunking_eval = ChunkingEvaluator(
    pipeline=ChunkNorrisPipeline(),
    chunkers=[
        PageChunker(),
        None # <-- We pass "None" as a chunker to also use the pipeline's default chunker
        ],
    retrievers=[
        DenseRetriever(
            model=model,
            description="Dense retriever with Snowflake/snowflake-arctic-embed-m-v2.0"
            )
        ],
    rerankers=[
        None, # <-- to try without reranker
        CrossEncoderReranker(
            n_to_rerank=200,
            description="test with reranking top 100 chunks."
            ),
        CrossEncoderReranker(
            n_to_rerank=200,
            description="test with reranking top 200 chunks."
            )
    ]
)
# Results will be saved by default in "./results" folder
chunking_eval.evaluate_chunking(queries_dataset, pdf_filepaths)

The parsing and chunking step of the evaluation may take some time. 

Consequently, you may want to reuse the chunked obtained from a previous run and run a new evaluation using another embedding model, **without rerunning the parsing and chunking**. 

In that case you can just reuse the obtained chunks. In that case you can use the following snippet.

In [ ]:
# Set new pipeline
chunking_eval = ChunkingEvaluator(
    pipeline=ChunkNorrisPipeline(),
    chunkers=[
        PageChunker(),
        None # <-- We pass "None" as a chunker to also use the pipeline's default chunker
        ],
    retrievers=[
        HybridRetriever(
            retrievers=[
                DenseRetriever(
                    model=model,
                    description="Dense retriever with Snowflake/snowflake-arctic-embed-m-v2.0"
                    ),
                BM25Retriever(),
                ],
            description="Hybrid retriever with SF-arctic-m-v2 + BM25",
            ),
        ],
    rerankers=[
        None, # <-- to try without reranker
        CrossEncoderReranker(
            n_to_rerank=200,
            description="test with reranking top 100 chunks."
            ),
        MixBreadAIReranker(description="Mxbai retriever with default config.")
    ]
)

# reload the chunks.json file as a datasetdict
chunking_eval.evaluate_chunking(
    queries_dataset=queries_dataset,
    path_to_chunks="results/timestamp_of_prev_exp/chunks.json"
    )